# 00 — Train and test the "green" membrane panel

This package trains ilastik on **one marker channel only** — here, Green - 488 (Erika small beads' macrophage marker). No nuclear channel exists in this dataset at all, so unlike the `red` panel's notebook, this one never touches `cell_tracker.nuclear_segmentation` — instances are split directly from the membrane mask's shape via distance-transform watershed (`cell_tracker.instance_segmentation.cells_from_membrane_mask`).

**History:** this panel originally reused `red`'s Red-561-trained classifier applied to the Green-488 channel. That generalized poorly on real data (near-saturated probabilities, over-classifying almost the whole frame as membrane), so it's being trained independently instead — this notebook walks through that.

This notebook: (1) exports single-channel Green-488 TIFFs for GUI training (already done once — `green/tiffs/` has 20 from the test file), (2) walks through training in the ilastik GUI, (3) runs the full pipeline (ilastik membrane probability + distance-transform watershed) against the test file and shows QC.

In [ ]:
import sys
from pathlib import Path

from cell_tracker import (
    discover_flat_files, list_panels, load_panel, scaffold_panel,
    run_segmentation_pipeline,
)

# This package's own panel library -- panels live as subfolders of
# the package itself (../ from this notebooks/ folder).
PANELS_ROOT = Path('..').resolve()
PANEL_NAME = 'green'

# The membrane/cytoplasm marker this panel is trained on.
MEMBRANE_CHANNEL = {'name': 'Green - 488', 'wavelength_nm': 488, 'index': None}
# No nuclear channel in this dataset -- unlike the red panel's notebook,
# there's nothing to read separately in Python here.

# Erika small beads' test file (the only project currently using this panel).
TEST_DIR = Path('../../../../Erika small beads/data/test').resolve()

print('Panels currently available:', list_panels(PANELS_ROOT) or '(none yet)')

In [ ]:
if PANEL_NAME in list_panels(PANELS_ROOT):
    panel = load_panel(PANELS_ROOT, PANEL_NAME)
    print('Loaded existing panel:', panel['panel_name'])
else:
    panel_dir = scaffold_panel(
        PANELS_ROOT, PANEL_NAME,
        channel_name=MEMBRANE_CHANNEL['name'],
        wavelength_nm=MEMBRANE_CHANNEL['wavelength_nm'],
        channel_index=MEMBRANE_CHANNEL['index'],
        description='Cytoplasm/membrane marker (Green - 488) ONLY panel -- the "membrane segmentation" for Erika small beads.',
    )
    print(f'Scaffolded new panel.yaml at {panel_dir} -- now:')
    print('  1. Open ilastik -> New Project -> Pixel Classification')
    print(f'  2. Save it as {panel_dir / "membrane_segmentation.ilp"}')
    print('  3. See this panel README.md for the rest of the training checklist')
    print('Re-run this cell after saving the .ilp to load it instead.')

# NOTE: this cell will raise FileNotFoundError ("panel.yaml exists but its
# ilastik project doesn't") until you've actually saved
# membrane_segmentation.ilp into green/ via the GUI steps below -- that's
# expected the first time through.

## 1. Export single-channel membrane TIFFs for GUI training

`green/tiffs/` already has 20 `..._488.tif` files exported from Erika small beads' test file (one per point) -- ready to load straight into ilastik. Re-run this cell only if you want to regenerate them or export from additional data.

In [ ]:
from tqdm.auto import tqdm
import tifffile

from cell_tracker import (
    discover_flat_files, extract_channels,
    get_point_names, load_nd2, iter_points, make_sample_id,
)

TIFFS_DIR = Path('.').resolve() / 'tiffs'
TIFFS_DIR.mkdir(parents=True, exist_ok=True)

files_df = discover_flat_files(TEST_DIR)

exported = []
for file_row in files_df.itertuples():
    point_names = get_point_names(file_row.nd2_path)
    arr, channel_meta, axes = load_nd2(file_row.nd2_path)
    for point_index, point_label, sub_arr, _ in tqdm(
        list(iter_points(arr, axes, point_names)), desc=file_row.filename
    ):
        sample_id = make_sample_id(file_row.file_id, point_label)
        imgs = extract_channels(sub_arr, channel_meta, {'membrane': MEMBRANE_CHANNEL}, {'max_project': True})
        out_path = TIFFS_DIR / f'{sample_id}_488.tif'
        tifffile.imwrite(out_path, imgs['membrane'].astype('uint16'))
        exported.append(out_path)

print(f'Exported {len(exported)} single-channel Green-488 TIFF(s) -> {TIFFS_DIR}')

## 2. Train in the ilastik GUI

1. Open ilastik -> **New Project -> Pixel Classification** (skip if you already created/loaded the project above).
2. Save it as `membrane_segmentation.ilp` in `cell segmentation/ilastik/cell tracker/green/` (the path printed above).
3. In Input Data, **"Add New" -> "Add separate image(s)..."** and select several `..._488.tif` files from `tiffs/` -- each becomes its own lane. No stacking dialog needed, this is a single channel.
4. Create exactly 3 labels, **in this order** (ilastik assigns indices 0, 1, 2... in creation order -- must match `panel.yaml`'s `ilastik.pixel_classes`): `membrane`, `background`, `cell_contact`.
5. Brush-label a handful of clear examples of each class across a few images, train, and check the live prediction overlay:
   - `membrane`: inside a cell body.
   - `background`: clearly empty space -- brush clean, unambiguous examples so the model has real contrast to learn from.
   - `cell_contact`: the thin boundary line where two touching cells' edges press together -- this is the class that lets the pipeline split touching cells from ilastik's own judgment instead of guessing geometrically (see `panel.yaml`'s `ilastik.contact_class` and this panel's README for how it's used).
6. Set Prediction Export source to **Probabilities**, save.

Then re-run the panel-loading cell above (cell 2) and continue below.

## 3. Test the full pipeline against the test file

Runs: extract Green-488 channel -> ilastik headless (membrane probability map) -> distance-transform watershed (no nuclear channel, no seed channel needed) -> QC panel for the first point. `nuclear_channel=None` (the default) is what switches `run_segmentation_pipeline` onto this pathway instead of the nucleus-seeded one `red`'s notebook uses.

In [ ]:
ILASTIK_EXECUTABLE = '/Applications/ilastik-1.4.1.post1-arm64-OSX.app/Contents/ilastik-release/run_ilastik.sh'

SCRATCH_DIR = Path('../scratch/green').resolve()

manifest, manifest_path = run_segmentation_pipeline(
    files_df=files_df,
    panel=panel,
    nuclear_channel=None,
    ilastik_executable=ILASTIK_EXECUTABLE,
    work_dir=SCRATCH_DIR,
    zstack_cfg={'max_project': True},
    qc_dir=SCRATCH_DIR / 'qc',
    qc_first_only=True,
)

print(f'Segmented {len(manifest)} point(s). Manifest -> {manifest_path}')
manifest[['replicate', 'condition', 'filename', 'point_label', 'membrane_area_fraction', 'segmentation_path']]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

qc_pngs = sorted((SCRATCH_DIR / 'qc').glob('*_qc.png'))
print(f'{len(qc_pngs)} QC panel(s) written (first point only, by default):')
for p in qc_pngs:
    print(' -', p)

if qc_pngs:
    img = mpimg.imread(qc_pngs[0])
    fig, ax = plt.subplots(figsize=(10, 15))
    ax.imshow(img)
    ax.axis('off')
    plt.show()

## Tuning

Manual knobs, all in `panel.yaml`'s `segmentation:` block (see this panel's `README.md` for full guidance): `smoothing_sigma` (off by default here -- don't blur the contact boundary), `probability_threshold`, `min_cell_size_px`, `min_seed_distance_px`, `contact_class`/`contact_probability_threshold` (the touching-cell boundary class), `fill_holes` (off by default -- incompatible with `contact_class`, see README). Adjust, then re-run the pipeline cell above with `force_segment=True` to re-segment without re-running ilastik.

Once QC looks right, this panel is ready — Erika small beads' `config.yaml` already points at panel `green`.